# MVP 2 — Risco, Custo e Tempo para Implantação de Nova Unidade de Armazenamento em Nuvem

**Aluno:** João Pedro Lima de Carvalho  
**Matrícula:** 231013402  
**Disciplina:** Sistemas de Suporte à Decisão

---

## 1. Definição do problema

Uma empresa de armazenamento de dados em nuvem vai implantar uma **nova unidade** (data center
regional). Antes de aprovar o projeto, a diretoria precisa de três respostas:

1. **Custo** — quanto o custo final tende a se desviar do orçamento aprovado?
2. **Tempo** — quantos dias de atraso esperar em relação ao cronograma planejado?
3. **Risco** — qual a probabilidade de a fase estourar custo ou prazo de forma relevante?

| Item | Definição |
|---|---|
| **Alvo 1 — Custo** | `fator_custo` = custo final ÷ orçamento base → **Regressão** |
| **Alvo 2 — Tempo** | `atraso_dias` = término real − término planejado → **Regressão** |
| **Alvo 3 — Risco** | estouro de custo > 10% **ou** atraso > 90 dias → **Classificação** |
| **Unidade de análise** | 1 registro = 1 **fase** de um projeto de capital |
| **Métricas** | MAE e R² (regressão); Acurácia, AUC e Recall (classificação) |

### Critério de sucesso (definido ANTES da modelagem)

Os três modelos precisam superar com folga um baseline ingênuo (média / classe majoritária).
**Expectativa calibrada:** em dados reais de projeto, R² entre 0,20 e 0,45 no estouro de custo
é um bom resultado. Se algum modelo passar de 0,90, a primeira hipótese é vazamento, não
mérito — a Seção 7 testa exatamente isso.

### Premissas

1. **Transferibilidade.** Um data center é, na maior parte do orçamento e do cronograma, obra
   civil e infraestrutura elétrica: terreno, licenciamento, obra, subestação, climatização.
   Projetos de capital públicos são o proxy real e auditável mais próximo disponível.
2. **Estabilidade temporal.** Padrões de estouro de 2003–2023 seguem válidos como referência.
3. **Sem ajuste inflacionário**, por serem usadas razões (adimensionais) e não valores absolutos.
4. **A fase é a unidade correta** de análise: o orçamento e o cronograma da base são definidos
   por fase, não por projeto inteiro.

### Restrições de seleção dos dados

- Somente fases com `Project Status Name = Complete` (única condição para existir término real).
- Orçamento base e custo final válidos e positivos.
- **Orçamento base ≥ US$ 10.000** — abaixo disso a razão é dominada por arredondamento em
  itens administrativos e gera fatores absurdos (a base tem casos de 41×).
- Datas de início, término planejado e término real presentes e coerentes.

---

## 2. Setup

In [ ]:
import warnings, re, unicodedata
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, roc_auc_score, recall_score,
                             confusion_matrix, classification_report, roc_curve)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Setup concluído.")

In [ ]:
# Carga — no Colab, envie 'capital-project-schedules-and-budgets.csv'
CAMINHO = "capital-project-schedules-and-budgets.csv"
try:
    from google.colab import files
    CAMINHO = list(files.upload().keys())[0]
except Exception:
    pass

bruto = pd.read_csv(CAMINHO, dtype=str, encoding="utf-8-sig")
bruto.columns = [c.strip() for c in bruto.columns]
print(f"Base bruta: {bruto.shape[0]:,} linhas x {bruto.shape[1]} colunas")
print(f"Projetos (DSF) distintos: {bruto['DSF Number(s)'].nunique():,}")
bruto.head(4)

## 3. Origem e trilha de auditoria da base

### Base utilizada

**SCA Capital Project Schedules and Budgets** — New York City School Construction Authority,
publicada no NYC Open Data e espelhada no Kaggle.

Cada linha é uma **fase** de um projeto de capital real (construção e reforma de escolas
públicas de Nova York) executado entre 2003 e 2023. O dicionário de dados oficial define:

| Coluna | Significado (dicionário oficial) |
|---|---|
| `Project Budget Amount` | Orçamento base aprovado por fase → **linha de base de custo** |
| `Final Estimate of Actual Costs...` | Estimativa final na conclusão da fase → **custo realizado** |
| `Project Phase Planned End Date` | Data em que a fase foi **originalmente** programada → **linha de base de prazo** |
| `Project Phase Actual End Date` | Data em que a fase realmente terminou → **prazo realizado** |
| `Total Phase Actual Spending Amount` | Despesa acumulada realizada |
| `DSF Number(s)` | Identificador do projeto no Plano de Cinco Anos |

### Por que esta base, e não uma sintética

Bases sintéticas de gestão de projeto são abundantes, mas nelas o alvo é gerado por fórmula:
o modelo recupera a fórmula e produz R² próximo de 1 sem qualquer descoberta. Esta base
contém **planejado contra realizado de obras efetivamente executadas**. O R² será menor —
e é esse o ponto: mede-se a dificuldade real de prever desempenho de projeto.

### Trilha de auditoria

| Candidata | Situação | Decisão |
|---|---|---|
| Bases sintéticas de "project management" (Kaggle) | Alvo gerado por fórmula; correlação quase determinística com uma única variável | **Descartada** — não sustenta conclusão de negócio |
| Bases de sensores de obra (IoT sintético) | Sem par planejado/realizado | **Descartada** — impossível derivar estouro |
| **SCA Capital Project Schedules and Budgets** | Planejado e realizado reais, procedência governamental, dicionário oficial | **Adotada** |

## 4. Diagnóstico da base

Antes de limpar, dois diagnósticos que determinam todo o desenho do trabalho.

In [ ]:
# 4.1 Códigos-sentinela: valores não-numéricos ocupando colunas numéricas e de data
SENTINELAS = {"PNS", "FTK", "IEH", "DOES", "DIIR"}

print("Códigos-sentinela encontrados por coluna:\n")
for c in ["Project Status Name", "Project Phase Actual Start Date", "Project Phase Planned End Date",
          "Project Phase Actual End Date", "Project Budget Amount"]:
    achados = sorted(set(bruto[c].dropna().unique()) & SENTINELAS)
    n = bruto[c].isin(SENTINELAS).sum()
    print(f"  {c:<38} {n:>6,} linhas  {achados}")

print("\nInterpretação (confrontada com o dicionário e com os tipos de projeto):")
print("  PNS  = Phase Not Started — fase ainda não iniciada, sem datas nem custo realizado")
print("  DIIR = projetos DIIT-RESOA — orçamento controlado fora deste dataset")
print("  IEH / DOES / FTK = tipos com orçamento gerido por outro programa")
print("\nEsses códigos NÃO são erro de dados: são marcadores de ausência estruturada.")
print("Tratá-los como texto e convertê-los para nulo é obrigatório — 'PNS' viraria")
print("categoria numérica se passasse direto para o modelo.")

In [ ]:
# 4.2 Estrutura de painel: a base NÃO é uma linha por projeto
n_linhas, n_proj = len(bruto), bruto["DSF Number(s)"].nunique()
print(f"Linhas: {n_linhas:,} | Projetos distintos (DSF): {n_proj:,} | Fases por projeto: {n_linhas/n_proj:.2f}")
print("\nDistribuição de fases:")
display(bruto["Project Phase Name"].value_counts().to_frame("linhas"))

print("\nConsequência metodológica: fases do mesmo projeto compartilham escola, distrito,")
print("tipo de obra e equipe. Tratá-las como observações independentes vaza informação")
print("entre treino e teste. Toda validação deste notebook usa GroupKFold agrupado por DSF.")

## 5. Limpeza e preparação

Funil documentado, com a contagem de linhas restantes a cada etapa.

In [ ]:
def para_numero(s):
    return pd.to_numeric(s.where(~s.fillna("").isin(SENTINELAS)), errors="coerce")

def para_data(s):
    return pd.to_datetime(s.where(~s.fillna("").isin(SENTINELAS)),
                          format="%m/%d/%Y", errors="coerce")

d = bruto.copy()
d["orcamento_base"] = para_numero(d["Project Budget Amount"])
d["custo_final"]    = para_numero(d["Final Estimate of Actual Costs Through End of Phase Amount"])
d["gasto_real"]     = para_numero(d["Total Phase Actual Spending Amount"])
d["dt_inicio"]      = para_data(d["Project Phase Actual Start Date"])
d["dt_plan_fim"]    = para_data(d["Project Phase Planned End Date"])
d["dt_real_fim"]    = para_data(d["Project Phase Actual End Date"])

PISO_ORCAMENTO = 10_000
funil = [("base bruta", len(d))]

c = d[d["Project Status Name"] == "Complete"];                      funil.append(("fase concluída (Complete)", len(c)))
c = c[c["orcamento_base"].notna() & c["custo_final"].notna()];      funil.append(("orçamento e custo final válidos", len(c)))
c = c[c["orcamento_base"] > 0];                                     funil.append(("orçamento base > 0", len(c)))
c = c[c["orcamento_base"] >= PISO_ORCAMENTO];                       funil.append((f"orçamento base >= US$ {PISO_ORCAMENTO:,}", len(c)))
c = c[c["dt_inicio"].notna() & c["dt_plan_fim"].notna() & c["dt_real_fim"].notna()]
funil.append(("datas de início, plano e realizado", len(c)))
c = c[c["dt_real_fim"] >= c["dt_inicio"]];                          funil.append(("coerência término >= início", len(c)))

display(pd.DataFrame(funil, columns=["etapa", "linhas restantes"]))
print(f"\nAproveitamento: {len(c)/len(d)*100:.1f}% das linhas | {c['DSF Number(s)'].nunique():,} projetos distintos")
print(f"\nO piso de US$ {PISO_ORCAMENTO:,} é a decisão de limpeza mais consequente deste MVP:")
print("sem ele, orçamentos de três dígitos produzem fatores de estouro de até 41x que")
print("dominam qualquer métrica de erro sem representar risco real de projeto.")
df = c.copy()

## 6. Construção dos alvos

Os três alvos são **derivados**, não vêm prontos. Os limiares do alvo de risco são
declarados aqui, antes de qualquer modelagem.

In [ ]:
LIMIAR_CUSTO, LIMIAR_PRAZO = 1.10, 90   # +10% de custo OU +90 dias de atraso

df["fator_custo"]  = df["custo_final"] / df["orcamento_base"]
df["atraso_dias"]  = (df["dt_real_fim"] - df["dt_plan_fim"]).dt.days
df["duracao_plan"] = (df["dt_plan_fim"] - df["dt_inicio"]).dt.days
df["ano_inicio"]   = df["dt_inicio"].dt.year

df["alto_risco"] = ((df["fator_custo"] > LIMIAR_CUSTO) |
                    (df["atraso_dias"] > LIMIAR_PRAZO)).astype(int)

resumo = pd.DataFrame({
    "fator_custo": df["fator_custo"].describe(percentiles=[.05,.25,.5,.75,.95,.99]),
    "atraso_dias": df["atraso_dias"].describe(percentiles=[.05,.25,.5,.75,.95,.99]),
})
display(resumo.round(2))

print(f"Assimetria do fator de custo : {df['fator_custo'].skew():.2f}")
print(f"Assimetria do atraso         : {df['atraso_dias'].skew():.2f}")
print(f"\nFases que estouraram custo (>1,0) : {(df['fator_custo']>1).mean()*100:.1f}%")
print(f"Fases que atrasaram (>0 dias)     : {(df['atraso_dias']>0).mean()*100:.1f}%")
print(f"Fases classificadas ALTO RISCO    : {df['alto_risco'].mean()*100:.1f}%")

print("\nResultado contraintuitivo e importante: a MEDIANA do fator de custo fica ABAIXO de 1.")
print("A SCA define o orçamento base de forma conservadora e a maioria das fases fecha abaixo")
print("dele. O risco não está no centro da distribuição — está na CAUDA, onde poucas fases")
print("estouram várias vezes o orçamento. É por isso que média e mediana enganam aqui.")

## 7. Auditoria de vazamento

Antes de treinar, o teste que decide a validade de todo o resto: existe alguma variável que
entrega a resposta?

In [ ]:
candidatas = ["orcamento_base", "gasto_real", "duracao_plan", "ano_inicio", "custo_final"]
corr = pd.DataFrame({
    "corr_fator_custo": [df[c].corr(df["fator_custo"]) for c in candidatas],
    "corr_atraso":      [df[c].corr(df["atraso_dias"]) for c in candidatas],
}, index=candidatas)
display(corr.round(3))

print("VEREDITO:\n")
print("  custo_final  -> É O NUMERADOR do alvo. Exclusão obrigatória.")
print("  gasto_real   -> Despesa acumulada REALIZADA. Só existe DEPOIS da fase terminar.")
print("                  Usar como preditor é vazamento temporal: no momento em que a")
print("                  diretoria precisa decidir, esse número ainda não existe.")
print("  orcamento_base, duracao_plan, ano_inicio -> conhecidos no PLANEJAMENTO. Válidos.")

In [ ]:
# Demonstração quantitativa do vazamento: mesmo modelo, com e sem 'gasto_real'
prova = df.dropna(subset=["gasto_real", "duracao_plan"]).copy()
y_prova = np.log(prova["fator_custo"].clip(lower=0.01))
g_prova = prova["DSF Number(s)"]
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)

prova_vaz = []
for rotulo, cols in [("COM gasto_real (vazamento)", ["orcamento_base","duracao_plan","ano_inicio","gasto_real"]),
                     ("SEM gasto_real (correto)",   ["orcamento_base","duracao_plan","ano_inicio"])]:
    X = prova[cols].fillna(prova[cols].median())
    pred = cross_val_predict(rf, X, y_prova, cv=GroupKFold(5), groups=g_prova, n_jobs=1)
    prova_vaz.append({"cenário": rotulo, "R² (log)": r2_score(y_prova, pred),
                 "MAE fator": mean_absolute_error(np.exp(y_prova), np.exp(pred))})

prova_vaz = pd.DataFrame(prova_vaz)
display(prova_vaz)
print(f"\nInflação de R² causada pelo vazamento: {prova_vaz.loc[0,'R² (log)'] - prova_vaz.loc[1,'R² (log)']:+.3f}")
print("\nO cenário com vazamento produz um número muito melhor e completamente inútil:")
print("prevê o custo final usando o quanto já foi gasto. Todo o restante do notebook")
print("usa exclusivamente variáveis disponíveis no momento do planejamento.")

## 8. Análise exploratória

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 9))

sns.histplot(df["fator_custo"].clip(0, 3), bins=50, ax=ax[0,0], color="#2b6cb0")
ax[0,0].axvline(1, color="r", ls="--", label="orçamento")
ax[0,0].set(title="Fator de custo (recorte 0–3)", xlabel="custo final ÷ orçamento"); ax[0,0].legend()

sns.histplot(df["atraso_dias"].clip(-400, 800), bins=50, ax=ax[0,1], color="#c05621")
ax[0,1].axvline(0, color="r", ls="--")
ax[0,1].set(title="Atraso (recorte −400 a 800 dias)", xlabel="dias")

ordem = df.groupby("Project Phase Name")["fator_custo"].median().sort_values().index
sns.boxplot(data=df, y="Project Phase Name", x="fator_custo", order=ordem,
            ax=ax[0,2], showfliers=False, palette="crest")
ax[0,2].axvline(1, color="r", ls="--"); ax[0,2].set(title="Fator de custo por fase", ylabel="")

sns.boxplot(data=df, y="Project Phase Name", x="atraso_dias", order=ordem,
            ax=ax[1,0], showfliers=False, palette="flare")
ax[1,0].axvline(0, color="r", ls="--"); ax[1,0].set(title="Atraso por fase", ylabel="")

ax[1,1].scatter(df["orcamento_base"], df["fator_custo"].clip(0,5), alpha=.25, s=12, color="#2f855a")
ax[1,1].set(xscale="log", title="Porte do orçamento vs. estouro",
            xlabel="orçamento base (US$, log)", ylabel="fator de custo")
ax[1,1].axhline(1, color="r", ls="--")

risco_tipo = (df.groupby("Project Type")["alto_risco"].agg(["mean","size"])
                .query("size >= 30").sort_values("mean", ascending=False))
sns.barplot(x=risco_tipo["mean"]*100, y=risco_tipo.index, ax=ax[1,2], palette="rocket")
ax[1,2].set(title="Taxa de alto risco por tipo (mín. 30 fases)", xlabel="% alto risco", ylabel="")

plt.tight_layout(); plt.show()

## 9. Viés de composição — o achado central deste MVP

Para existir data real de término, a fase precisa estar **concluída**. Mas a taxa de conclusão
não é uniforme entre as fases: fases iniciais terminam rápido e entram na amostra; fases de
construção demoram anos e ainda estão em andamento na data-corte.

O resultado é **censura à direita**: as fases mais longas e mais atrasadas são justamente as
que ficam de fora.

In [ ]:
todas = bruto.copy()
composicao = (todas.groupby("Project Phase Name")["Project Status Name"]
                   .value_counts(normalize=True).unstack().fillna(0) * 100)
composicao["total_linhas"] = todas["Project Phase Name"].value_counts()
display(composicao.round(1).sort_values("Complete", ascending=False))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ordem_c = composicao.sort_values("Complete", ascending=False).index
composicao.loc[ordem_c, ["Complete","In-Progress","PNS"]].plot(
    kind="barh", stacked=True, ax=ax[0], color=["#2f855a","#d69e2e","#a0aec0"])
ax[0].set(title="Composição por situação, dentro de cada fase", xlabel="% das linhas", ylabel="")
ax[0].legend(title="")

perfil = df.groupby("Project Phase Name").agg(
    n=("atraso_dias","size"), atraso_mediano=("atraso_dias","median"),
    fator_mediano=("fator_custo","median")).sort_values("atraso_mediano")
sns.barplot(x=perfil["atraso_mediano"], y=perfil.index, ax=ax[1], palette="vlag")
ax[1].axvline(0, color="k", lw=1)
ax[1].set(title="Atraso mediano por fase (amostra concluída)", xlabel="dias", ylabel="")
plt.tight_layout(); plt.show()
display(perfil)

In [ ]:
# Quantificando a censura à direita
data_corte = max(todas_dt for todas_dt in [df["dt_real_fim"].max(), df["dt_plan_fim"].max()])
em_curso = bruto[bruto["Project Status Name"] == "In-Progress"].copy()
em_curso["plan"] = para_data(em_curso["Project Phase Planned End Date"])
em_curso = em_curso[em_curso["plan"].notna()]
vencidas = em_curso["plan"] < data_corte
atraso_ja = (data_corte - em_curso.loc[vencidas, "plan"]).dt.days

print(f"Data-corte inferida da base: {data_corte.date()}\n")
print(f"Fases em andamento com prazo definido : {len(em_curso):,}")
print(f"  já vencidas na data-corte           : {vencidas.sum():,} ({vencidas.mean()*100:.1f}%)")
print(f"  atraso mínimo já acumulado (mediana): {atraso_ja.median():,.0f} dias")
print(f"\nComparação: atraso mediano da amostra concluída = {df['atraso_dias'].median():.0f} dias")

print("\n" + "="*74)
print("CONSEQUÊNCIA PARA A DECISÃO")
print("="*74)
print("As fases excluídas da modelagem não são um recorte aleatório: são as mais atrasadas.")
print("Qualquer estimativa de atraso derivada apenas das fases concluídas é, por construção,")
print("OTIMISTA. Como um data center é dominado por Construction e CM,F&E — as fases com")
print("MENOR taxa de conclusão e MAIOR atraso mediano — o modelo geral subestimaria o risco.")
print("\nDECISÃO DE PROJETO: reportar resultados ESTRATIFICADOS por fase e, na camada de")
print("decisão (Seção 13), calibrar o data center com as fases de obra, não com a média geral.")

## 10. Atributos e pré-processamento

Só entram variáveis **conhecidas no momento do planejamento**. `gasto_real` e `custo_final`
ficam de fora por decisão da Seção 7.

In [ ]:
df["distrito"] = df["Project Geographic District"].astype(str).str.strip()
df["borough"]   = df["Project Building Identifier"].astype(str).str[0].map(
    {"K":"Brooklyn","M":"Manhattan","Q":"Queens","X":"Bronx","R":"Staten Island"}).fillna("Outro")
df["log_orcamento"] = np.log10(df["orcamento_base"])

# Palavras-chave do escopo: sinal de complexidade extraído do texto livre
desc = df["Project Description"].fillna("").str.upper()
for chave, termos in {"eletrica":["ELECTRIC","LIGHT","POWER"], "hvac":["HVAC","HEAT","BOILER","CLIMATE"],
                      "estrutural":["ROOF","FACADE","STRUCT","MASONRY","PARAPET"],
                      "seguranca":["SAFETY","SECURITY","FIRE","ALARM"],
                      "acessibilidade":["ACCESS","RAMP","ELEVATOR","PATH OF TRAVEL"]}.items():
    df[f"esc_{chave}"] = desc.str.contains("|".join(termos)).astype(int)

CAT = ["Project Type", "Project Phase Name", "borough"]
NUM = ["log_orcamento", "duracao_plan", "ano_inicio",
       "esc_eletrica", "esc_hvac", "esc_estrutural", "esc_seguranca", "esc_acessibilidade"]

for c in CAT:
    freq = df[c].value_counts()
    df[c] = df[c].where(~df[c].isin(freq[freq < 20].index), "OUTROS")

print("Categóricos:", CAT)
print("Numéricos  :", NUM)
print(f"\nRegistros: {len(df):,} | Projetos (grupos): {df['DSF Number(s)'].nunique():,}")

try:
    ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10, sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preproc = ColumnTransformer([
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="NA")), ("ohe", ohe)]), CAT),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
])

X = df[CAT + NUM].copy()
grupos = df["DSF Number(s)"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
i_tr, i_te = next(gss.split(X, groups=grupos))
print(f"\nTreino: {len(i_tr):,} fases | Teste: {len(i_te):,} fases")
print(f"Projetos em comum entre treino e teste: "
      f"{len(set(grupos.iloc[i_tr]) & set(grupos.iloc[i_te]))} (deve ser 0)")

## 11. Modelo 1 — Custo

Alvo em **log** do fator de custo: o estouro é multiplicativo e a distribuição tem cauda
pesada (assimetria acima de 10). Sem o log, poucas fases extremas dominariam o ajuste.

In [ ]:
y_custo = np.log(df["fator_custo"].clip(lower=0.05, upper=df["fator_custo"].quantile(0.99)))

modelos_reg = {
    "Baseline (média)": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=400, min_samples_leaf=12, max_features=0.5,
                                           random_state=RANDOM_STATE, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
                                                          min_samples_leaf=15, l2_regularization=1.0,
                                                          random_state=RANDOM_STATE),
}

def avaliar_reg(y, X, alvo_nome, escala_log=True):
    linhas = {}
    Xtr, ytr, gtr = X.iloc[i_tr], y.iloc[i_tr], grupos.iloc[i_tr]
    for nome, mod in modelos_reg.items():
        pipe = Pipeline([("prep", clone(preproc)), ("mod", clone(mod))])
        oof = cross_val_predict(pipe, Xtr, ytr, cv=GroupKFold(5), groups=gtr, n_jobs=1)
        if escala_log:
            mae = mean_absolute_error(np.exp(ytr), np.exp(oof))
        else:
            mae = mean_absolute_error(ytr, oof)
        linhas[nome] = {"R² (CV)": r2_score(ytr, oof), "MAE (CV)": mae}
    return pd.DataFrame(linhas).T.sort_values("R² (CV)", ascending=False)

cv_custo = avaliar_reg(y_custo, X, "custo")
print("VALIDAÇÃO CRUZADA — GroupKFold 5 folds, agrupado por projeto\n")
display(cv_custo.round(4))
MELHOR_CUSTO = cv_custo.index[0] if cv_custo.index[0] != "Baseline (média)" else cv_custo.index[1]
print(f"Melhor: {MELHOR_CUSTO}")

In [ ]:
pipe_custo = Pipeline([("prep", clone(preproc)), ("mod", clone(modelos_reg[MELHOR_CUSTO]))])
pipe_custo.fit(X.iloc[i_tr], y_custo.iloc[i_tr])
pred_te = pipe_custo.predict(X.iloc[i_te])
y_te = y_custo.iloc[i_te]

res_custo = {
    "R² (log)": r2_score(y_te, pred_te),
    "MAE (fator)": mean_absolute_error(np.exp(y_te), np.exp(pred_te)),
    "MAE (% do orçamento)": mean_absolute_error(np.exp(y_te), np.exp(pred_te)) * 100,
}
display(pd.DataFrame([res_custo], index=["Teste (não visto)"]).round(4))

gap = r2_score(y_custo.iloc[i_tr], pipe_custo.predict(X.iloc[i_tr])) - res_custo["R² (log)"]
print(f"Gap de R² treino−teste: {gap:.3f}", "→ OVERFITTING" if gap > 0.2 else "→ ajuste equilibrado")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].scatter(np.exp(y_te), np.exp(pred_te), alpha=.4, s=14, color="#2b6cb0")
lim = [0, 3]; ax[0].plot(lim, lim, "r--"); ax[0].set(xlim=lim, ylim=lim,
    xlabel="fator real", ylabel="fator previsto", title=f"Custo — real vs. previsto ({MELHOR_CUSTO})")
sns.histplot(np.exp(y_te) - np.exp(pred_te), bins=40, ax=ax[1], color="#c05621")
ax[1].axvline(0, color="r", ls="--"); ax[1].set(title="Resíduos do fator de custo", xlabel="erro")
plt.tight_layout(); plt.show()

## 12. Modelo 2 — Tempo

In [ ]:
y_prazo = df["atraso_dias"].clip(df["atraso_dias"].quantile(0.01), df["atraso_dias"].quantile(0.99))
cv_prazo = avaliar_reg(y_prazo, X, "prazo", escala_log=False)
print("VALIDAÇÃO CRUZADA — atraso em dias\n")
display(cv_prazo.round(3))
MELHOR_PRAZO = cv_prazo.index[0] if cv_prazo.index[0] != "Baseline (média)" else cv_prazo.index[1]

pipe_prazo = Pipeline([("prep", clone(preproc)), ("mod", clone(modelos_reg[MELHOR_PRAZO]))])
pipe_prazo.fit(X.iloc[i_tr], y_prazo.iloc[i_tr])
pp = pipe_prazo.predict(X.iloc[i_te])
res_prazo = {"R²": r2_score(y_prazo.iloc[i_te], pp),
             "MAE (dias)": mean_absolute_error(y_prazo.iloc[i_te], pp),
             "RMSE (dias)": np.sqrt(mean_squared_error(y_prazo.iloc[i_te], pp))}
display(pd.DataFrame([res_prazo], index=["Teste (não visto)"]).round(2))

print(f"\nO MAE de {res_prazo['MAE (dias)']:.0f} dias é a informação acionável: é o piso de folga")
print("que a empresa deve embutir no cronograma contratual de cada fase.")

## 13. Modelo 3 — Risco (classificação)

O erro aqui é **assimétrico**: classificar como seguro um projeto que vai estourar custa
muito mais caro que o contrário. Por isso o limiar de decisão é deslocado de 0,5.

In [ ]:
y_risco = df["alto_risco"]
modelos_clf = {
    "Baseline (classe majoritária)": DummyClassifier(strategy="most_frequent"),
    "Regressão Logística": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
}
linhas = []
for nome, mod in modelos_clf.items():
    pipe = Pipeline([("prep", clone(preproc)), ("mod", clone(mod))])
    oof = cross_val_predict(pipe, X.iloc[i_tr], y_risco.iloc[i_tr], cv=GroupKFold(5),
                            groups=grupos.iloc[i_tr], method="predict", n_jobs=1)
    linhas.append({"Modelo": nome, "Acurácia (CV)": accuracy_score(y_risco.iloc[i_tr], oof),
                   "Recall (CV)": recall_score(y_risco.iloc[i_tr], oof)})
cv_risco = pd.DataFrame(linhas).set_index("Modelo").sort_values("Recall (CV)", ascending=False)
display(cv_risco.round(4))

pipe_risco = Pipeline([("prep", clone(preproc)), ("mod", clone(modelos_clf["Random Forest"]))])
pipe_risco.fit(X.iloc[i_tr], y_risco.iloc[i_tr])
prob = pipe_risco.predict_proba(X.iloc[i_te])[:, 1]
yr_te = y_risco.iloc[i_te]

res_risco = {"Acurácia": accuracy_score(yr_te, prob >= 0.5), "AUC": roc_auc_score(yr_te, prob),
             "Recall @0,5": recall_score(yr_te, prob >= 0.5)}
display(pd.DataFrame([res_risco], index=["Teste (não visto)"]).round(4))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
fpr, tpr, _ = roc_curve(yr_te, prob)
ax[0].plot(fpr, tpr, color="#2b6cb0", lw=2); ax[0].plot([0,1],[0,1],"r--")
ax[0].set(xlabel="Falso positivo", ylabel="Verdadeiro positivo",
          title=f"Curva ROC (AUC = {res_risco['AUC']:.3f})")

sns.heatmap(confusion_matrix(yr_te, prob >= 0.5), annot=True, fmt="d", cmap="Blues", ax=ax[1],
            xticklabels=["prev. seguro","prev. risco"], yticklabels=["real seguro","real risco"])
ax[1].set(title="Matriz de confusão (limiar 0,50)")

lim_grid = np.linspace(0.2, 0.8, 40)
ax[2].plot(lim_grid, [recall_score(yr_te, prob >= t) for t in lim_grid], label="Recall")
ax[2].plot(lim_grid, [accuracy_score(yr_te, prob >= t) for t in lim_grid], label="Acurácia")
ax[2].axvline(0.35, color="r", ls="--", label="limiar sugerido 0,35")
ax[2].set(xlabel="limiar de decisão", title="Trade-off do limiar"); ax[2].legend()
plt.tight_layout(); plt.show()

LIMIAR_OP = 0.35
print(f"Com limiar {LIMIAR_OP}: recall {recall_score(yr_te, prob>=LIMIAR_OP):.3f} | "
      f"acurácia {accuracy_score(yr_te, prob>=LIMIAR_OP):.3f}")
print("Baixar o limiar aumenta os falsos alarmes e reduz os projetos de risco que passam")
print("despercebidos. Para decisão de capital, é o trade-off correto.")

## 14. Importância das variáveis

In [ ]:
imp = permutation_importance(pipe_custo, X.iloc[i_te], y_te, n_repeats=10,
                             random_state=RANDOM_STATE, n_jobs=-1,
                             scoring="neg_mean_absolute_error")
tab = (pd.DataFrame({"atributo": X.columns, "importancia": imp.importances_mean,
                     "desvio": imp.importances_std})
       .sort_values("importancia", ascending=False))
plt.figure(figsize=(9, 5))
o = tab.iloc[::-1]
plt.barh(o["atributo"], o["importancia"], xerr=o["desvio"], color="#2b6cb0", alpha=.85, capsize=3)
plt.title("Importância por permutação — modelo de custo"); plt.xlabel("queda de desempenho")
plt.tight_layout(); plt.show()
display(tab.round(4))

print("Leitura de negócio: nenhuma variável concentra a maior parte da importância.")
print("Isso é o oposto do padrão de base sintética (uma variável dominante) e indica")
print("que o estouro em projetos reais é multicausal — coerente com a literatura.")

## 15. Da previsão à decisão — orçamento P80 e reserva de contingência

Um modelo de regressão não responde "quanto reservar". Para isso é preciso a **distribuição**
do estouro, não a média. Esta seção usa a distribuição empírica das fases de obra
(`Construction`, `CM,F&E`) — as relevantes para um data center — numa simulação de Monte Carlo.

In [ ]:
FASES_OBRA = ["Construction", "CM,F&E", "CM,Art,F&E", "CM"]
obra = df[df["Project Phase Name"].isin(FASES_OBRA)]
amostra_fc     = obra["fator_custo"].dropna().values
amostra_atraso = obra["atraso_dias"].dropna().values
print(f"Calibração com {len(obra)} fases de obra reais "
      f"(mediana do fator: {np.median(amostra_fc):.3f}, atraso: {np.median(amostra_atraso):.0f} dias)")

# EAP simplificada do data center (orçamento base em R$)
EAP = pd.DataFrame([
    ("Terreno e licenciamento",      4_000_000),
    ("Projeto executivo",            2_500_000),
    ("Obra civil",                  18_000_000),
    ("Infraestrutura elétrica",     15_000_000),
    ("Climatização",                 9_000_000),
    ("Conectividade e fibra",        3_500_000),
    ("Hardware de TI",              22_000_000),
    ("Comissionamento e go-live",    3_000_000),
], columns=["pacote", "orcamento_base"])
BASE_TOTAL = EAP["orcamento_base"].sum()

N, RHO = 10_000, 0.35
rng = np.random.default_rng(RANDOM_STATE)
custos = np.empty(N); atrasos = np.empty(N)
n_pac = len(EAP)

for i in range(N):
    # fator comum + idiossincrático: pacotes de um mesmo projeto atrasam juntos
    u = stats.norm.cdf(np.sqrt(RHO)*rng.standard_normal() +
                       np.sqrt(1-RHO)*rng.standard_normal(n_pac))
    fatores = np.quantile(amostra_fc, u)
    custos[i]  = (EAP["orcamento_base"].values * fatores).sum()
    atrasos[i] = np.quantile(amostra_atraso, u).max()   # o pacote mais atrasado manda

p50c, p80c, p90c = np.percentile(custos, [50, 80, 90])
p50t, p80t, p90t = np.percentile(atrasos, [50, 80, 90])

fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
for a, dados, p50, p80, tit, un in [(ax[0], custos/1e6, p50c/1e6, p80c/1e6, "Custo total simulado", "R$ mi"),
                                     (ax[1], atrasos, p50t, p80t, "Atraso total simulado", "dias")]:
    sns.histplot(dados, bins=60, ax=a, color="#2b6cb0", alpha=.7)
    a.axvline(p50, color="#d69e2e", ls="--", lw=2, label=f"P50 = {p50:,.0f}")
    a.axvline(p80, color="#c53030", ls="--", lw=2, label=f"P80 = {p80:,.0f}")
    a.set(title=tit, xlabel=un); a.legend()
plt.tight_layout(); plt.show()

print("="*70)
print("  RECOMENDAÇÃO ORÇAMENTÁRIA")
print("="*70)
print(f"  Orçamento base (determinístico) : R$ {BASE_TOTAL/1e6:>8,.1f} mi")
print(f"  Custo P50 (50% de chance)       : R$ {p50c/1e6:>8,.1f} mi")
print(f"  Custo P80  <-- recomendado      : R$ {p80c/1e6:>8,.1f} mi")
print(f"  Custo P90                       : R$ {p90c/1e6:>8,.1f} mi")
print(f"  RESERVA DE CONTINGÊNCIA (P80)   : R$ {(p80c-BASE_TOTAL)/1e6:>8,.1f} mi "
      f"({(p80c/BASE_TOTAL-1)*100:.1f}% sobre a base)")
print("-"*70)
print(f"  Buffer de cronograma (P80)      : {p80t:>8,.0f} dias")
print("="*70)
print("\nOrçar pela média equivale a aceitar ~50% de chance de estouro. O P80 é o padrão")
print("de mercado para infraestrutura: financia o risco sem imobilizar capital em excesso.")

## 16. Playbook de decisão — como a empresa deve se programar

Como as saídas dos modelos viram rotina de gestão.

In [ ]:
def avaliar_pacote(tipo, fase, orcamento, duracao_planejada, borough="Brooklyn",
                   ano=2026, escopo=None):
    '''Avalia um pacote de trabalho: fator de custo, atraso e probabilidade de risco.'''
    reg = {"Project Type": tipo, "Project Phase Name": fase, "borough": borough,
           "log_orcamento": np.log10(max(orcamento, 1)), "duracao_plan": duracao_planejada,
           "ano_inicio": ano}
    for k in ["eletrica","hvac","estrutural","seguranca","acessibilidade"]:
        reg[f"esc_{k}"] = int(k in (escopo or []))
    e = pd.DataFrame([reg])[CAT + NUM]

    faixa_orc = (df["log_orcamento"].min(), df["log_orcamento"].max())
    extrapola = not (faixa_orc[0] <= reg["log_orcamento"] <= faixa_orc[1])

    fator  = float(np.exp(pipe_custo.predict(e)[0]))
    atraso = float(pipe_prazo.predict(e)[0])
    p      = float(pipe_risco.predict_proba(e)[0, 1])
    acao = ("COMITÊ DE APROVAÇÃO + garantia contratual" if p >= LIMIAR_OP
            else "Fluxo padrão de aprovação")

    print("="*66)
    print(f"  {fase} | {tipo} | orçamento base R$ {orcamento:,.0f}")
    print("-"*66)
    print(f"  Fator de custo previsto  : {fator:.3f}  "
          f"(orçamento de referência R$ {orcamento*fator:,.0f})")
    print(f"  Atraso previsto          : {atraso:+.0f} dias")
    print(f"  Folga contratual mínima  : {atraso + res_prazo['MAE (dias)']:.0f} dias "
          f"(previsão + MAE do modelo)")
    print(f"  Probabilidade de risco   : {p*100:.1f}%  (limiar operacional {LIMIAR_OP*100:.0f}%)")
    print(f"  >> {acao}")
    if extrapola:
        print("  [!] Orçamento fora da faixa observada no treino — previsão pouco confiável")
    print("="*66)
    return {"fator": fator, "atraso": atraso, "prob_risco": p}

_ = avaliar_pacote("SCA CIP", "Construction", 18_000_000, 540,
                   escopo=["eletrica","hvac"])

### Rotina recomendada

1. **Triagem.** Rodar o modelo de risco em cada pacote de trabalho antes da contratação.
   Acima do limiar operacional de 0,35, exigir comitê e garantia contratual.
2. **Orçamento de referência.** Multiplicar o orçamento base pelo fator de custo previsto.
   Proposta muito abaixo dessa referência é sinal de alerta, não de oportunidade.
3. **Folga de cronograma.** Usar a previsão de atraso somada ao MAE do modelo como piso de
   folga contratual por fase. Nunca contratar a data otimista.
4. **Reserva de contingência.** Provisionar o P80 da simulação, não a média. Revisar a cada
   marco concluído, recalibrando com os dados realizados.
5. **Foco de controle na obra.** As fases de construção têm a maior taxa de atraso e a menor
   taxa de conclusão dentro do prazo. É onde alocar gestão, não em Scope e Design.
6. **Reavaliação periódica.** Retreinar conforme a empresa acumular projetos próprios,
   substituindo progressivamente a base de referência externa.

---

## 17. Conclusões, limitações e próximos passos

### Conclusões

1. **Vazamento evitado e quantificado.** Usar `gasto_real` como preditor produziria R² muito
   superior e inútil: prevê o custo final a partir do que já foi gasto. A Seção 7 mede a
   inflação e justifica a exclusão.

2. **O risco está na cauda, não no centro.** A mediana do fator de custo fica abaixo de 1 —
   a maioria das fases fecha abaixo do orçamento base. Mas a assimetria acima de 10 mostra
   uma cauda em que poucas fases estouram várias vezes o previsto. Média e mediana escondem
   exatamente o que a gestão precisa enxergar.

3. **Viés de composição é a maior ameaça à validade.** Só fases concluídas têm término real,
   e a taxa de conclusão despenca nas fases de obra (Construction 12,7%, Purch & Install 2,0%,
   contra Scope 67,8%). As fases excluídas são as mais atrasadas. Qualquer estimativa geral é
   otimista por construção — daí a estratificação por fase e a calibração da simulação com
   as fases de obra.

4. **Estouro é multicausal.** Nenhuma variável concentra a importância, ao contrário do que
   ocorre em bases sintéticas. É um problema genuinamente multivariado.

5. **A previsão pontual não basta.** O que sustenta a decisão de capital é o P80 e a reserva
   de contingência, produzidos pela simulação da Seção 15.

### Limitações

- **Transferibilidade.** Obras escolares públicas de Nova York como proxy de data center
  privado no Brasil: contexto regulatório, cambial e de cadeia de suprimentos diferentes.
  Não há risco de importação de hardware na base de origem.
- **Censura à direita** não corrigida formalmente. A abordagem rigorosa seria análise de
  sobrevivência com dados censurados; o MVP trata o problema por estratificação e o
  documenta explicitamente.
- **Sem correção inflacionária** entre 2003 e 2023, mitigada pelo uso de razões.
- **Ausência de variáveis de execução** (empreiteiro, clima, licenciamento, cadeia de
  suprimentos), que a literatura aponta como determinantes de atraso.
- **Poder explicativo modesto** — o esperado para projetos reais, e a razão pela qual a
  saída é uma faixa com reserva, não um número único.

### Próximos passos

| # | Ação | Ganho |
|---|---|---|
| 1 | Modelo de sobrevivência (Cox / AFT) tratando fases em andamento como censuradas | Alto — corrige o viés na raiz |
| 2 | Substituir progressivamente a base externa por projetos da própria empresa | Alto |
| 3 | Incorporar variáveis de sítio: energia, licenciamento, classificação sísmica | Médio |
| 4 | Modelagem conjunta (multi-output) de custo, prazo e risco | Médio |
| 5 | Empacotar `avaliar_pacote()` em API ou Streamlit para o time de planejamento | Alto |

---

**João Pedro Lima de Carvalho** — Matrícula 231013402  
*MVP 2 desenvolvido para a disciplina de Sistemas de Suporte à Decisão.*

In [ ]:
# Exportação dos resultados e das bases tratadas
df.to_csv("base_tratada_completa.csv", index=False)
df[CAT + NUM + ["fator_custo","atraso_dias","alto_risco","DSF Number(s)"]].to_csv(
    "base_modelagem.csv", index=False)

import json as _json
resultados = {
    "n_registros": int(len(df)), "n_projetos": int(df["DSF Number(s)"].nunique()),
    "modelo_custo": {"nome": MELHOR_CUSTO, **{k: float(v) for k, v in res_custo.items()}},
    "modelo_prazo": {"nome": MELHOR_PRAZO, **{k: float(v) for k, v in res_prazo.items()}},
    "modelo_risco": {"nome": "Random Forest", **{k: float(v) for k, v in res_risco.items()}},
    "simulacao": {"base": float(BASE_TOTAL), "P50": float(p50c), "P80": float(p80c),
                  "P90": float(p90c), "reserva_P80": float(p80c - BASE_TOTAL),
                  "buffer_dias_P80": float(p80t)},
}
with open("resultados_mvp2.json", "w", encoding="utf-8") as f:
    _json.dump(resultados, f, ensure_ascii=False, indent=2)
print("Arquivos gerados: base_tratada_completa.csv, base_modelagem.csv, resultados_mvp2.json")
print(_json.dumps(resultados, indent=2, ensure_ascii=False))